### Quickstart v2: Aggregators and Portfolio Optimizers

This notebook demonstrates the new pipeline:

- Models → Model Simulations
- Aggregators (combine model weights)
- Optional Portfolio Optimizers (asset-level, e.g., mean-variance)
- Portfolio Simulations

It uses small defaults and runs end-to-end.


In [1]:
import datetime as dt
import polars as pl

from trading_engine.core import (
    read_data,
    create_model_state,
    orchestrate_model_backtests,
    orchestrate_model_simulations,
    orchestrate_portfolio_aggregation,
    orchestrate_portfolio_optimizations,
    orchestrate_portfolio_simulations,
    run_full_backtest,
)
from trading_engine.models import MODELS
from trading_engine.aggregators import AGGREGATORS
from trading_engine.optimizers import PORTFOLIO_OPTIMIZERS

import datetime

print(f"polars version: {pl.__version__}")
print(f"available models: {list(MODELS.keys())[:5]} ...")
print(f"available aggregators: {list(AGGREGATORS.keys())}")
print(f"available portfolio optimizers: {list(PORTFOLIO_OPTIMIZERS.keys())}")


polars version: 1.32.0
available models: ['RXI_TLT_pml_10', 'GLD_USO_nml_10', 'IXJ_USO_pml_10', 'TLT_IXC_nml_10', 'IXJ_KXI_pml_10'] ...
available aggregators: ['equal_weight', 'min_avg_drawdown']
available portfolio optimizers: ['mean_variance']


In [2]:
# 1) experiment config (small)
universe = [
  "SPY-US", "SLV-US", "GLD-US", "TLT-US", "USO-US", "UNG-US", "IXJ-US",
  "KXI-US", "JXI-US", "IXG-US", "IXN-US", "RXI-US", "MXI-US", "EXI-US",
  "IXC-US", "IEI-US", "SHY-US", "BIL-US", "JPXN-US", "INDA-US", "MCHI-US",
  "EZU-US", "IBIT-US", "ETHA-US", "VIXY-US"
]
features = ["close_momentum_10", "natr_14"]
models   = ["RXI_TLT_pml_10", "GLD_USO_nml_10"]
aggregators = ["equal_weight"]
portfolio_optimizers = ["mean_variance"]  # try [] to skip optimizer stage

initial_value = 1_000_000
start_date = datetime.date(2020, 1, 1)
end_date   = datetime.date(2025, 8, 30)

universe, features, models, aggregators, portfolio_optimizers


(['SPY-US',
  'SLV-US',
  'GLD-US',
  'TLT-US',
  'USO-US',
  'UNG-US',
  'IXJ-US',
  'KXI-US',
  'JXI-US',
  'IXG-US',
  'IXN-US',
  'RXI-US',
  'MXI-US',
  'EXI-US',
  'IXC-US',
  'IEI-US',
  'SHY-US',
  'BIL-US',
  'JPXN-US',
  'INDA-US',
  'MCHI-US',
  'EZU-US',
  'IBIT-US',
  'ETHA-US',
  'VIXY-US'],
 ['close_momentum_10', 'natr_14'],
 ['RXI_TLT_pml_10', 'GLD_USO_nml_10'],
 ['equal_weight'],
 ['mean_variance'])

In [3]:
# 2) build model state + prices
lf = read_data()
model_state, prices = create_model_state(
    lf=lf,
    features=features,
    start_date=start_date,
    end_date=end_date,
    universe=universe,
)

model_state.head(), prices.head()


(shape: (5, 9)
 ┌────────────┬────────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬──────────┐
 │ date       ┆ ticker ┆ adjusted_o ┆ adjusted_h ┆ … ┆ adjusted_ ┆ volume_1d ┆ close_mom ┆ natr_14  │
 │ ---        ┆ ---    ┆ pen_1d     ┆ igh_1d     ┆   ┆ close_1d  ┆ ---       ┆ entum_10  ┆ ---      │
 │ date       ┆ str    ┆ ---        ┆ ---        ┆   ┆ ---       ┆ i64       ┆ ---       ┆ f64      │
 │            ┆        ┆ f64        ┆ f64        ┆   ┆ f64       ┆           ┆ f64       ┆          │
 ╞════════════╪════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
 │ 2020-01-02 ┆ BIL-US ┆ 80.637566  ┆ 80.64638   ┆ … ┆ 80.64638  ┆ 3167468   ┆ 0.07494   ┆ 0.018245 │
 │ 2020-01-03 ┆ BIL-US ┆ 80.64638   ┆ 80.64638   ┆ … ┆ 80.64638  ┆ 853176    ┆ 0.05734   ┆ 0.0174   │
 │ 2020-01-06 ┆ BIL-US ┆ 80.64638   ┆ 80.65523   ┆ … ┆ 80.637566 ┆ 2471905   ┆ 0.039726  ┆ 0.0179   │
 │ 2020-01-07 ┆ BIL-US ┆ 80.64638   ┆ 80.64638   ┆ … ┆ 80.637566 ┆ 

In [4]:
# 3) run model backtests + simulations
model_insights = orchestrate_model_backtests(
    model_state=model_state,
    models=models,
    universe=universe,
)

model_simulations = orchestrate_model_simulations(
    prices=prices,
    model_insights=model_insights,
    initial_value=initial_value,
)

list(model_insights.keys()), list(model_simulations.keys())


(['RXI_TLT_pml_10', 'GLD_USO_nml_10'], ['RXI_TLT_pml_10', 'GLD_USO_nml_10'])

In [5]:
# 4) aggregate model insights (new aggregator stage)
aggregated = orchestrate_portfolio_aggregation(
    model_insights=model_insights,
    backtest_results=model_simulations,
    universe=universe,
    aggregators=aggregators,
)

list(aggregated.keys())


['equal_weight']

In [6]:
# 5) optional: asset-level portfolio optimization (mean-variance)
optimized = {}
if portfolio_optimizers:
    optimized = orchestrate_portfolio_optimizations(
        prices=prices,
        aggregated_insights=aggregated,
        universe=universe,
        optimizers=portfolio_optimizers,
    )

final = optimized if optimized else aggregated
list(final.keys())


['mean_variance']

In [7]:
# 6) simulate the chosen portfolio
sims = orchestrate_portfolio_simulations(
    prices=prices,
    portfolio_insights=final,
    initial_value=initial_value,
)

sims


{'mean_variance': {'backtest_results': shape: (1_422, 9)
  ┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
  │ date      ┆ portfolio ┆ daily_ret ┆ daily_log ┆ … ┆ cumulativ ┆ drawdown  ┆ volume_tr ┆ daily_sl │
  │ ---       ┆ _value    ┆ urn       ┆ _return   ┆   ┆ e_log_ret ┆ ---       ┆ aded      ┆ ippage_c │
  │ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ urn       ┆ f64       ┆ ---       ┆ ost      │
  │           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆           ┆ f64       ┆ ---      │
  │           ┆           ┆           ┆           ┆   ┆ f64       ┆           ┆           ┆ f64      │
  ╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
  │ 2020-01-0 ┆ 1e6       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
  │ 2         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
  │ 2020-01-0 ┆ 

In [8]:
# 7) view summary metrics for the first portfolio
pname, presults = next(iter(sims.items()))
metrics = presults.get("backtest_metrics")
metrics


metric,value
str,f64
"""total_return""",-0.40613
"""annualized_return""",-0.08821
"""annualized_volatility""",0.121578
"""sharpe_ratio""",-0.725544
"""sortino_ratio""",-0.631957
…,…
"""num_weight_events""",1422.0
"""parsing_time_ms""",14.0
"""simulation_time_ms""",12.0


In [9]:
# 8) one-shot end-to-end via run_full_backtest
results = run_full_backtest(
    universe=universe,
    features=features,
    models=models,
    aggregators=aggregators,
    portfolio_optimizers=portfolio_optimizers,
    start_date=start_date,
    end_date=end_date,
    initial_value=initial_value,
)

results.keys()


dict_keys(['model_simulations', 'aggregation_results', 'aggregation_simulations', 'optimizer_results', 'optimizer_simulations'])

In [10]:
# 9) compare aggregated vs optimized metrics (if both present)
from IPython.display import display

agg_sims = results.get("aggregation_simulations", {})
opt_sims = results.get("optimizer_simulations", {})

if agg_sims:
    agg_name, agg_sim = next(iter(agg_sims.items()))
    print(f"Aggregator: {agg_name}")
    display(agg_sim.get("backtest_metrics"))
else:
    print("No aggregation simulations found.")

if opt_sims:
    opt_name, opt_sim = next(iter(opt_sims.items()))
    print(f"\nOptimizer: {opt_name}")
    display(opt_sim.get("backtest_metrics"))
else:
    print("\nNo optimizer simulations found.")


Aggregator: equal_weight


metric,value
str,f64
"""total_return""",0.782623
"""annualized_return""",0.107877
"""annualized_volatility""",0.098819
"""sharpe_ratio""",1.091666
"""sortino_ratio""",0.806158
…,…
"""num_weight_events""",1422.0
"""parsing_time_ms""",15.0
"""simulation_time_ms""",12.0



Optimizer: mean_variance


metric,value
str,f64
"""total_return""",-0.40613
"""annualized_return""",-0.08821
"""annualized_volatility""",0.121578
"""sharpe_ratio""",-0.725544
"""sortino_ratio""",-0.634854
…,…
"""num_weight_events""",1422.0
"""parsing_time_ms""",14.0
"""simulation_time_ms""",13.0
